# 📖 Notebook 8: Service Mesh — Istio for Traffic Management and Security

Welcome to the service mesh lab. In plain language, a **service mesh** gives your apps a smart helper layer for service-to-service communication. Instead of teaching every app how to do retries, encryption, traffic shaping, and detailed observability, the mesh handles much of that work beside your app.

In this lab, we will use Istio with the sample microservices in the `k8s-lab` namespace:
- `api-gateway` on port `8000`
- `user-service` on port `8001`
- `order-service` on port `8002`

A simple way to think about it is: **your app focuses on business logic, and the mesh focuses on how services talk safely and reliably.**

> **Prerequisites: Notebooks 01–03.** Istio is layered on top of the running `k8s-lab`
> services.
>
> **Resources**: the Istio `demo` profile installs istiod plus ingress and egress
> gateways, and adds a ~50–100 Mi sidecar to *every* pod in `k8s-lab`. Plan on
> **6 GB of cluster memory minimum**; 8 GB (Notebook 01's default) is comfortable. If
> Notebook 05's Prometheus stack is still running and pods start going `Pending`,
> `helm uninstall prometheus -n monitoring` first.

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

# `helm` is here because the capacity check below uninstalls notebook 05's
# Prometheus release to make room for Istio.
REQUIRED = ['kubectl', 'curl', 'helm']
INSTALL_HINTS = {
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
    'curl': 'preinstalled on macOS and most Linux distros; `apt install curl` otherwise',
    'helm': 'https://helm.sh/docs/intro/install/  (or `brew install helm`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

# A reachable cluster is required too -- `kubectl` alone is not enough.
probe = subprocess.run(
    ['kubectl', 'cluster-info'], capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(
        'No reachable Kubernetes cluster. Start the one from notebook 1:\n'
        '  minikube start --cpus=4 --memory=6144 --driver=docker\n'
        f'kubectl said: {probe.stderr.strip()[:300]}'
    )

print('Preflight OK:', ', '.join(REQUIRED), '+ cluster reachable')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────
import json
import subprocess
import time

NS = "k8s-lab"


def kget(*args, ns=NS):
    cmd = ["kubectl", "get", *args, "-o", "json"] + (["-n", ns] if ns else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("kubectl failed: " + r.stderr.strip()[:400])
    return json.loads(r.stdout)


def ready_pods(selector, ns=NS):
    return [p for p in kget("pods", "-l", selector, ns=ns)["items"]
            if "deletionTimestamp" not in p["metadata"]
            and any(c["type"] == "Ready" and c["status"] == "True"
                    for c in p["status"].get("conditions", []))]


def wait_until(predicate, timeout=170, interval=5, what="condition"):
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    raise AssertionError(f"timed out after {timeout}s waiting for {what}")


def poll(predicate, timeout, interval=10):
    """wait_until, but returns None instead of raising -- for waits long enough
    that they are split across two cells."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    return None


def deployments_ready(ns, names):
    counts = {}
    for name in names:
        r = subprocess.run(["kubectl", "get", "deployment", name, "-n", ns,
                            "-o", "json"], capture_output=True, text=True)
        counts[name] = (json.loads(r.stdout)["status"].get("readyReplicas", 0)
                        if r.returncode == 0 else 0)
    print("  ready:", counts)
    return counts if all(v >= 1 for v in counts.values()) else None


DEPLOYMENTS = ("api-gateway", "user-service", "order-service")


def container_names(pod):
    """Every container in the pod, INCLUDING init containers.

    This matters more than it looks. On Kubernetes 1.29+ Istio injects the proxy
    as a NATIVE SIDECAR: an entry in `spec.initContainers` with
    `restartPolicy: Always`, not in `spec.containers`. It still starts before the
    app, still runs for the pod's whole life, and still counts in the READY
    column -- but code that only looks at `spec.containers` concludes there is no
    sidecar at all. (Native sidecars are also what finally fixed "my Job never
    completes because the proxy is still running": the kubelet shuts them down
    once the main containers exit.)"""
    return ([c["name"] for c in pod["spec"].get("initContainers", [])]
            + [c["name"] for c in pod["spec"]["containers"]])


def sidecar_state():
    """(pods with a sidecar, total Ready pods) across the three lab Deployments."""
    with_sidecar = total = 0
    for name in DEPLOYMENTS:
        for pod in ready_pods(f"app={name}"):
            total += 1
            with_sidecar += "istio-proxy" in container_names(pod)
    return with_sidecar, total


def await_sidecars(want_injected, budget=150):
    """Poll until all six lab pods are (or are not) carrying a sidecar.

    Replacing six pods with `maxUnavailable: 0` is a couple of minutes of work,
    which is longer than a single notebook cell should block for -- hence the
    budget, and hence this being called from two consecutive cells."""
    deadline = time.time() + budget
    while time.time() < deadline:
        injected, total = sidecar_state()
        # `>= 6` rather than `== 6`: re-running the notebook can leave the canary
        # Deployment from the previous pass around, and its pods also carry
        # `app: user-service`.
        done = total >= 6 and (injected == total if want_injected else injected == 0)
        print(f"  {int(time.time() - deadline + budget):>3}s  ready={total}/6  "
              f"with sidecar={injected}")
        if done:
            return True
        time.sleep(10)
    return False


def node_headroom():
    """Rough free memory on the node in MiB, as the kubelet sees it.

    `kubectl top node --no-headers` prints:
        NAME  CPU(cores)  CPU(%)  MEMORY(bytes)  MEMORY(%)
    so the memory figure is field 3, not field 2 -- an easy off-by-one that
    silently reads a percentage as a byte count."""
    r = subprocess.run(["kubectl", "top", "node", "--no-headers"],
                       capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.split():
        return None
    used_mib = int(r.stdout.split()[3].removesuffix("Mi"))
    cap = kget("nodes", ns=None)["items"][0]["status"]["allocatable"]["memory"]
    return int(cap.removesuffix("Ki")) // 1024 - used_mib


free_mb = node_headroom()
print(f"cluster memory headroom (as the kubelet sees it): ~{free_mb} MiB"
      if free_mb else "(metrics-server not up)")

# Make room before installing Istio. The demo profile is istiod plus two gateways,
# and every pod in k8s-lab gains a ~50-100 Mi sidecar; notebook 05's
# kube-prometheus-stack is the other big resident. Nothing from here on reads from
# Prometheus, so the two do not need to coexist -- and on a 6 GB cluster the
# failure mode when they do is not a tidy `Pending` pod. With the docker driver the
# kubelet reports the HOST's memory as the node's capacity, so it never sees memory
# pressure and never evicts anything; Docker stalls the whole node container
# instead and the cluster simply stops answering.
#
# Set FREE_MONITORING = False if you would rather keep Grafana and have the RAM.
FREE_MONITORING = True

monitoring_exists = subprocess.run(["kubectl", "get", "namespace", "monitoring"],
                                   capture_output=True).returncode == 0
if monitoring_exists and FREE_MONITORING:
    print("\nremoving notebook 05's monitoring stack to make room for Istio")
    print("(reinstall it with the `helm upgrade --install prometheus ...` from nb 05)")
    subprocess.run(["helm", "uninstall", "prometheus", "-n", "monitoring",
                    "--ignore-not-found"], capture_output=True)
    subprocess.run(["kubectl", "delete", "namespace", "monitoring",
                    "--ignore-not-found"], capture_output=True)
elif monitoring_exists:
    print("\n⚠️  the monitoring stack is still installed and FREE_MONITORING is False.")
    print("   If pods start going Pending, or the cluster stops responding entirely,")
    print("   run: helm uninstall prometheus -n monitoring && kubectl delete ns monitoring")

print("\nhelpers ready")

## Learning Objectives

By the end of this notebook, you will be able to:

- explain what a service mesh is and why teams use one
- describe Istio sidecars in beginner-friendly terms
- enable mTLS for service-to-service encryption
- split traffic between service versions for canary releases
- add retries and timeouts without changing application code
- view service relationships with Kiali

## 🛠️ Setup

Before starting:

1. Make sure your Kubernetes cluster is running.
2. Make sure the `k8s-lab` namespace and sample services are already deployed.
3. Istio demo installs several extra components, so a cluster with **6 GB of memory or more** is strongly recommended.
4. Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

This notebook uses shell commands so you can see the mesh setup step by step.

In [ ]:
!kubectl cluster-info
!kubectl get nodes -o wide
!kubectl get deployments -n k8s-lab

## What is a Service Mesh?

A service mesh is a **dedicated infrastructure layer for service-to-service communication**. That sentence sounds big, so let us shrink it:

- your services still send HTTP requests like normal
- small helper proxies sit next to the services
- those proxies handle networking features for you

This matters because distributed systems get complicated quickly. Even a small app can need encryption, retries, routing rules, and visibility into failures. A mesh centralizes those concerns.

### 🧪 Practical Exercise
Look at the three services in `k8s-lab` and imagine adding retries and encryption to every service by hand. Which approach feels easier to maintain: changing all apps or adding shared traffic rules in one place?

## 🔀 Pod A → Sidecar Proxy → Sidecar Proxy → Pod B

```text
+-----------+      +------------------+      +------------------+      +-----------+
|   Pod A   | ---> | Sidecar Proxy A  | ---> | Sidecar Proxy B  | ---> |   Pod B   |
| app code  |      | handles traffic  |      | handles traffic  |      | app code  |
+-----------+      +------------------+      +------------------+      +-----------+
```

The key idea is that the app does not need to know every networking trick. The sidecar proxies take care of many cross-cutting concerns.

## Why use a mesh?

A service mesh can provide these features **without changing your app code**:

- **mTLS** for encrypted service-to-service traffic
- **retries** when a request fails briefly
- **timeouts** so requests do not hang forever
- **traffic splitting** for canary releases
- **observability** so you can see service relationships and traffic flow

### 🧪 Practical Exercise
Pick one feature from the list and explain what problem it solves. For example, why might a timeout be better than waiting forever?

In [ ]:
!kubectl get svc -n k8s-lab
!kubectl get pods -n k8s-lab

## 1) Install Istio

We will install the Istio demo profile. It includes enough features for a learning lab.

⚠️ Security Warning: the quick-start command below pipes a downloaded script into `sh` because that matches the common Istio lab flow. In a stricter environment, download the script first, inspect it, and only then run it.

### 🧪 Practical Exercise
Before running the install, predict what new namespace will appear and what kind of pods you expect to see there after Istio is installed.

In [ ]:
# `curl -L https://istio.io/downloadIstio | sh -` fetches ~100 MB and unpacks it
# into ./istio-<version>/ next to this notebook. Re-running the notebook should
# not re-download it, so look for an existing copy first.
import glob
import os

existing = sorted(glob.glob("./istio-*/bin/istioctl"))
if existing:
    print("reusing the Istio release already unpacked here:",
          os.path.dirname(os.path.dirname(existing[-1])))
    print("(delete that directory if you want to pick up a newer release)")
else:
    print("no local Istio release found -- downloading (~100 MB)")
    download = subprocess.run(
        "curl -L https://istio.io/downloadIstio | sh -",
        shell=True, text=True)
    if download.returncode != 0:
        raise RuntimeError("the Istio download script failed; see the output above")

In [ ]:
# Resolve istioctl once and keep it in a Python variable. The original
# `!export PATH=... && istioctl ...` pattern has to be repeated in every cell,
# because each `!` line runs in its own throwaway shell -- exports never persist.
candidates = sorted(glob.glob("./istio-*/bin/istioctl"))
if not candidates:
    raise RuntimeError(
        "istioctl not found. Did the download cell above succeed? "
        "Expected ./istio-<version>/bin/istioctl"
    )
ISTIOCTL = os.path.abspath(candidates[-1])
ISTIO_DIR = os.path.dirname(os.path.dirname(ISTIOCTL))
print("istioctl:", ISTIOCTL)
!$ISTIOCTL version --remote=false

In [ ]:
# `istioctl install` is idempotent: re-running it reconciles to the same state.
# It returns once the control plane's objects are applied.
!$ISTIOCTL install --set profile=demo -y

In [ ]:
# istiod, the ingress gateway and the egress gateway are three fresh images on a
# new cluster. Bounded poll, continued in the next cell.
ISTIO_DEPLOYMENTS = ("istiod", "istio-ingressgateway", "istio-egressgateway")

print("waiting for the Istio control plane:")
state = (deployments_ready("istio-system", ISTIO_DEPLOYMENTS)
         or poll(lambda: deployments_ready("istio-system", ISTIO_DEPLOYMENTS), timeout=160))
print("all up" if state else "still pulling -- the next cell keeps waiting")

In [ ]:
state = (deployments_ready("istio-system", ISTIO_DEPLOYMENTS)
         or poll(lambda: deployments_ready("istio-system", ISTIO_DEPLOYMENTS), timeout=170))
!kubectl get pods -n istio-system

# istiod is the control plane. Without it there is no certificate authority, no
# xDS configuration and no sidecar injection webhook -- every later section of
# this notebook would fail in a different and confusing way.
assert state, (
    "the Istio control plane never became Ready. `kubectl get pods -n istio-system` "
    "and `kubectl describe` will say why; on a fresh cluster it is usually a slow "
    "image pull, and re-running this cell continues the wait."
)

# The injection webhook is the specific piece the next section depends on.
hooks = kget("mutatingwebhookconfiguration", ns=None)["items"]
assert any("sidecar-injector" in h["metadata"]["name"] for h in hooks), \
    "istiod is up but the sidecar-injector webhook was not registered"
print("\n✅ Istio control plane installed, injection webhook registered")

### ⚠️ First: Pod Security Standards vs Istio's init container

`k8s-lab` is labelled `pod-security.kubernetes.io/enforce: baseline` (notebook 02 applied
that with the namespace). Sidecar injection is about to fail against it, and the error is
worth understanding rather than working around blindly:

```text
Error creating: pods "user-service-..." is forbidden: violates PodSecurity
"baseline:latest": non-default capabilities (container "istio-init" must not
include "NET_ADMIN", "NET_RAW" in securityContext.capabilities.add)
```

By default Istio injects an **init container** (`istio-init`) that rewrites the pod's
iptables rules so that all traffic goes through the sidecar. Rewriting iptables needs
`NET_ADMIN` and `NET_RAW`, and `baseline` does not allow either. Nothing is misconfigured
— two correct security mechanisms genuinely disagree.

There are exactly two honest resolutions:

| Option | What it costs |
|---|---|
| Relax the namespace to `enforce: privileged` | Every pod in the namespace loses baseline protection, not just Istio's init container. Fine for a lab, wrong for production. |
| Install the **Istio CNI plugin** (`istioctl install --set components.cni.enabled=true`) | The iptables rewrite moves to a privileged DaemonSet that runs once per node, so workload pods need no elevated capabilities at all and the namespace can stay `restricted`. This is the production answer. |

We take the first for the lab, because the CNI plugin needs node-specific paths that vary
by distribution, and we **put the label back** in the cleanup cell so notebooks 9 and 10
inherit the namespace they expect. If you take anything from this section, take the second
row: on a real cluster with Pod Security enforced, `components.cni.enabled=true` is what
you want.

In [ ]:
# Relax Pod Security on k8s-lab so the istio-init container is admitted. The
# cleanup cell at the bottom restores `baseline`.
before = kget("namespace", NS, ns=None)["metadata"]["labels"]
print("PSS labels before:", {k: v for k, v in before.items() if "pod-security" in k})

!kubectl label namespace k8s-lab pod-security.kubernetes.io/enforce=privileged --overwrite

after = kget("namespace", NS, ns=None)["metadata"]["labels"]
assert after.get("pod-security.kubernetes.io/enforce") == "privileged", \
    f"the enforce label did not change: {after}"
print("PSS labels after: ", {k: v for k, v in after.items() if "pod-security" in k})

## 2) Enable sidecar injection

Istio adds sidecar proxies when a namespace is labeled for injection. After labeling the
namespace, we restart the workloads so new pods are created with the sidecars attached.

### Where the sidecar actually lives

`kubectl get pods` will show `2/2` — but the two are not both in `spec.containers`. On
Kubernetes 1.29 and newer, Istio injects the proxy as a **native sidecar**: an entry in
`spec.initContainers` carrying `restartPolicy: Always`.

```bash
kubectl get pod -n k8s-lab -l app=api-gateway \
  -o jsonpath='{.items[0].spec.initContainers[*].name}'
# istio-init istio-proxy
```

That is not cosmetic. A native sidecar starts **before** the app container (so the app
never makes a request before the proxy is listening) and, crucially, the kubelet
terminates it once the main containers exit — which is what finally fixed the old
"my Job stays Running forever because the proxy never stops" problem. It also means any
script that looks for the proxy in `spec.containers` will conclude there is no sidecar,
which is exactly the mistake the helper `container_names()` at the top of this notebook
exists to avoid.

The other init container, `istio-init`, is the short-lived one that rewrites the pod's
iptables rules — the one Pod Security objected to above.

### 🧪 Practical Exercise
After the restart, inspect the pod READY column. If a pod shows `2/2`, which two
containers are being counted, and which section of the pod spec is each declared in?

In [ ]:
# The label only affects pods created AFTER it is set -- existing pods keep
# running without a sidecar until they are replaced. Hence the rollout restart.
#
# NOTE the exact command: `kubectl rollout restart deployment -n k8s-lab` restarts
# every Deployment in the namespace. `... deployment --all` is NOT the same thing:
# `rollout restart` has no `--all` flag, so that form exits non-zero, and because
# a failing `!` line does not fail a notebook cell you would sail straight past it
# into a section that quietly never injected anything.
!kubectl label namespace k8s-lab istio-injection=enabled --overwrite
!kubectl rollout restart deployment -n k8s-lab

print("\nreplacing six pods, one per Deployment at a time (maxUnavailable: 0):")
finished = await_sidecars(want_injected=True, budget=150)
print("done" if finished else "still rolling -- the next cell keeps waiting")

In [ ]:
# Second half of the wait, then the check.
injected, total = sidecar_state()
if not (total >= 6 and injected == total):
    assert await_sidecars(want_injected=True, budget=170), \
        f"pods never came back with sidecars: {sidecar_state()}"

print()
# READY 2/2 = your container + the injected istio-proxy sidecar.
!kubectl get pods -n k8s-lab

# "2/2" is the claim; check it rather than eyeballing the column. A rollout that
# quietly kept the old, sidecar-less pods looks almost identical in `kubectl get`.
for name in DEPLOYMENTS:
    running = ready_pods(f"app={name}")
    assert len(running) >= 2, f"{name} has only {len(running)} ready pods"
    for pod in running:
        containers = container_names(pod)
        assert "istio-proxy" in containers, (
            f"{pod['metadata']['name']} has no sidecar: {containers}. If this failed "
            "with a PodSecurity error, the label change two cells up did not apply."
        )
print("\n✅ every pod in k8s-lab now runs its app container plus an istio-proxy sidecar")

## 3) Enforce mTLS

**mTLS** means **mutual TLS**. In simple terms, both sides of a connection prove who they are, and the traffic is encrypted. This helps protect traffic moving between services inside the cluster.

We will use a `PeerAuthentication` resource in `STRICT` mode so sidecars require mTLS.

⚠️ `STRICT` rejects **plaintext** traffic to these pods. Any workload without a sidecar —
including a `kubectl run` debug pod, and including Prometheus if it is scraping from
outside the mesh — stops being able to reach them. That is why the mesh default is
`PERMISSIVE`: it accepts both while you migrate, and you flip to `STRICT` only once
every client is injected.

### 🧪 Practical Exercise
Read the YAML and say what the word `STRICT` suggests. Is the mesh allowing plain text traffic, preferring encryption, or requiring encryption?

In [ ]:
%%writefile ./istio-peer-auth.yaml
# security.istio.io/v1 has been the stable version since Istio 1.21.
# v1beta1 is still served and identical in shape, so older clusters work too.
apiVersion: security.istio.io/v1
kind: PeerAuthentication
metadata:
  name: default
  namespace: k8s-lab
spec:
  mtls:
    # STRICT   -> sidecars REQUIRE mTLS; plaintext is rejected.
    # PERMISSIVE (the mesh default) -> accept both, so you can migrate gradually.
    # DISABLE  -> plaintext only.
    mode: STRICT

In [ ]:
!kubectl apply -f ./istio-peer-auth.yaml
!kubectl get peerauthentication -n k8s-lab

pa = kget("peerauthentication", "default")
assert pa["spec"]["mtls"]["mode"] == "STRICT", pa["spec"]
print("\n✅ PeerAuthentication default/STRICT applied to k8s-lab")

## 4) Verify mTLS

Istio has helper commands that can inspect how traffic is protected. This is useful because encryption is hard to prove just by looking at your application code.

### 🧪 Practical Exercise
Run the check below and look for evidence that the connection is using Istio-managed TLS instead of plain text traffic.

Look for two things in the output below:

1. `istioctl x describe pod` reporting the effective policy, e.g.
   `Effective PeerAuthentication: STRICT`.
2. A `default` entry under `proxy-config secret` — that is the workload's own X.509
   certificate, issued by istiod, with a SPIFFE URI SAN like
   `spiffe://cluster.local/ns/k8s-lab/sa/default`. mTLS authenticates *that identity*,
   which is why a mesh identity is stronger than an IP allow-list: IPs get reused, SPIFFE
   identities do not.

In [ ]:
# NOTE: `istioctl authn tls-check` was removed from istioctl years ago (it was
# deprecated back in Istio 1.5). These are the current equivalents.
POD = subprocess.run(
    ["kubectl", "get", "pod", "-n", "k8s-lab", "-l", "app=api-gateway",
     "-o", "jsonpath={.items[0].metadata.name}"],
    capture_output=True, text=True,
).stdout.strip()
print("inspecting pod:", POD)

# Prints the effective PeerAuthentication mode and which policies produced it.
!$ISTIOCTL x describe pod -n k8s-lab $POD

print("\n--- proof the sidecar actually holds an identity certificate ---")
!$ISTIOCTL proxy-config secret -n k8s-lab $POD

# Both claims from the markdown above, checked. `proxy-config secret -o json`
# gives the certificate chain the sidecar is actually using right now.
secrets = json.loads(subprocess.run(
    [ISTIOCTL, "proxy-config", "secret", "-n", NS, POD, "-o", "json"],
    capture_output=True, text=True).stdout)
names = [d.get("name") for d in secrets.get("dynamicActiveSecrets", [])]
assert "default" in names, \
    f"the sidecar holds no workload certificate (found {names}) -- mTLS cannot work"

described = subprocess.run(
    [ISTIOCTL, "x", "describe", "pod", "-n", NS, POD],
    capture_output=True, text=True).stdout
assert "STRICT" in described, \
    f"the effective PeerAuthentication is not STRICT:\n{described[:600]}"
print("\n✅ effective policy is STRICT and the sidecar holds an istiod-issued identity")

## 5) Traffic splitting for a canary release

A canary release sends a small percentage of traffic to a newer version while most users still hit the stable version. This reduces risk.

In this lab, we will:

1. label the existing `user-service` pods as `v1`
2. create a small `user-service-v2` deployment
3. route 90% of traffic to `v1` and 10% to `v2`

The sample FastAPI app returns the same payload from both versions, so you cannot tell
them apart from a response body. We measure the split the way you would in a real
investigation instead: send a few hundred requests, then count how many landed on each
version's pods by reading their access logs. (Kiali shows the same thing as a graph
later, and Istio's own `istio_requests_total` metric carries a `destination_version`
label for the same purpose.)

Note how the weights are enforced: the **client's** sidecar makes the routing decision
before the request ever leaves the calling pod. There is no proxy hop in the middle, and
the plain Kubernetes Service is still what resolves `user-service` to a set of endpoints —
Istio only changes which of those endpoints gets picked. Also note that this is *traffic*
splitting, not user splitting: with 10% weight, one user's ten requests land on v2 roughly
once each, they do not get a consistent version. Sticky canaries need
`http.match` on a header, or a `consistentHash` load balancer in the DestinationRule.

### 🧪 Practical Exercise
Before applying the canary, explain why sending only 10% of traffic to a new version is safer than sending 100% immediately.

In [ ]:
# Add a `version` label to the existing user-service pods so the v1 subset can
# select them. The Deployment's selector is `app: user-service` only, so adding a
# label to the pod template is safe -- selectors are immutable, pod labels are not.
#
# Changing the pod template is a full rollout, and with `maxUnavailable: 0` plus a
# sidecar to start on each new pod that is a couple of minutes -- more than one
# notebook cell should block for, so the wait continues in the next cell.
!kubectl patch deployment user-service -n k8s-lab --type merge -p '{"spec":{"template":{"metadata":{"labels":{"version":"v1"}}}}}'


def v1_labelled():
    ready = ready_pods("app=user-service,version=v1")
    print(f"  v1-labelled pods Ready: {len(ready)}/2")
    return ready if len(ready) == 2 else None


state = v1_labelled() or poll(v1_labelled, timeout=150)
print("done" if state else "still rolling -- the next cell keeps waiting")

In [ ]:
v1 = v1_labelled() or poll(v1_labelled, timeout=170)
!kubectl get pods -l app=user-service -n k8s-lab

assert v1, "the user-service pods never came back carrying version=v1"
assert all("istio-proxy" in container_names(p) for p in v1), \
    "the relabelled pods lost their sidecars"
print(f"\n✅ {len(v1)} pods now carry version=v1, with sidecars")

In [ ]:
%%writefile ./istio-canary.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: user-service-v2
  namespace: k8s-lab
spec:
  replicas: 1
  selector:
    matchLabels:
      app: user-service
      version: v2
  template:
    metadata:
      labels:
        app: user-service
        version: v2
    spec:
      containers:
        - name: user-service
          image: k8s-lab/user-service:latest
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8001
---
# networking.istio.io/v1 is stable since Istio 1.22; v1beta1 is still served.
# A DestinationRule names the subsets. Without it, `subset: v1` in the
# VirtualService below refers to nothing and Envoy returns 503 NR.
apiVersion: networking.istio.io/v1
kind: DestinationRule
metadata:
  name: user-service
  namespace: k8s-lab
spec:
  host: user-service
  subsets:
    - name: v1
      labels:
        version: v1
    - name: v2
      labels:
        version: v2
---
apiVersion: networking.istio.io/v1
kind: VirtualService
metadata:
  name: user-service
  namespace: k8s-lab
spec:
  hosts:
    - user-service
  http:
    - route:
        - destination:
            host: user-service
            subset: v1
          weight: 90
        - destination:
            host: user-service
            subset: v2
          weight: 10

In [ ]:
!kubectl apply -f ./istio-canary.yaml
!kubectl rollout status deployment/user-service-v2 -n k8s-lab --timeout=120s

# One more pod, one more sidecar to start. Bounded, so the cell cannot outlast a
# notebook runner's budget.
assert poll(lambda: len(ready_pods("app=user-service,version=v2")) == 1, timeout=60), \
    "the v2 canary pod never became Ready"
!kubectl get deployment user-service user-service-v2 -n k8s-lab
!kubectl get virtualservice,destinationrule -n k8s-lab

# A VirtualService pointing at a subset no DestinationRule defines is not an
# error -- Envoy just answers 503 NR for every request to it. So check that both
# subsets exist AND that each has running pods behind it before we split traffic.
dr = kget("destinationrule", "user-service")
assert {s["name"] for s in dr["spec"]["subsets"]} == {"v1", "v2"}, dr["spec"]["subsets"]

vs = kget("virtualservice", "user-service")
weights = {r["destination"]["subset"]: r["weight"] for r in vs["spec"]["http"][0]["route"]}
assert weights == {"v1": 90, "v2": 10}, f"unexpected traffic weights: {weights}"

for subset, want in (("v1", 2), ("v2", 1)):
    got = ready_pods(f"app=user-service,version={subset}")
    assert len(got) == want, f"subset {subset} has {len(got)} ready pods, expected {want}"
    for pod in got:
        assert "istio-proxy" in container_names(pod), \
            f"{pod['metadata']['name']} has no sidecar; it cannot participate in the mesh"
print(f"\n✅ subsets v1 (2 pods) and v2 (1 pod) are both serving, weights {weights}")

## 6) Add retries and timeouts

Traffic management is not only about percentages. You can also tell the mesh how patient to be and how many times to retry before giving up.

Below, we update the `VirtualService` so requests to `user-service` use a 2-second timeout and up to 3 retry attempts for certain transient failures.

### 🧪 Practical Exercise
Why might a retry help with a short network hiccup, but not with a permanent application bug?

In [ ]:
%%writefile ./istio-traffic-policy.yaml
apiVersion: networking.istio.io/v1
kind: VirtualService
metadata:
  name: user-service
  namespace: k8s-lab
spec:
  hosts:
    - user-service
  http:
    # `timeout` is the budget for the WHOLE request including retries.
    # perTryTimeout x attempts must fit inside it or the later attempts can
    # never happen: 3 x 1s = 3s > 2s here, so in practice you get ~2 tries.
    - timeout: 2s
      retries:
        attempts: 3
        perTryTimeout: 1s
        # Only retry conditions that are safe to repeat. Note what is NOT here:
        # 5xx in general, and anything on a non-idempotent verb -- retrying a
        # POST that already succeeded but whose response was lost duplicates it.
        retryOn: gateway-error,connect-failure,refused-stream
      route:
        - destination:
            host: user-service
            subset: v1
          weight: 90
        - destination:
            host: user-service
            subset: v2
          weight: 10

In [ ]:
!kubectl apply -f ./istio-traffic-policy.yaml
!kubectl get virtualservice user-service -n k8s-lab -o yaml | head -n 40

## 7) Observability with Kiali

Kiali is a dashboard that helps you see the mesh as a graph. This is one of the most beginner-friendly ways to understand what is talking to what.

### 🧪 Practical Exercise
After opening Kiali, try to identify the direction of requests between `api-gateway` and `user-service`. The goal is to connect the abstract mesh concepts to a picture.

In [ ]:
# Use the addons that ship WITH the Istio version we just downloaded. Pinning a
# release branch in a URL (the original `release-1.24/samples/addons/kiali.yaml`)
# breaks as soon as istioctl downloads a newer Istio.
#
# Kiali reads its graph data from Prometheus, so the Prometheus addon is not
# optional -- without it the graph renders empty. This is Istio's own lightweight
# Prometheus, separate from notebook 05's kube-prometheus-stack.
!kubectl apply -f $ISTIO_DIR/samples/addons/prometheus.yaml
!kubectl apply -f $ISTIO_DIR/samples/addons/kiali.yaml

# NOT `kubectl wait --for=condition=available`: that is satisfied as soon as the
# Deployment has its MINIMUM availability, which for a single-replica Deployment
# mid-rollout can be true while the new pod is still starting. readyReplicas is
# the number that means "a pod is actually serving". Bounded poll, continued in
# the next cell, because both images are fresh downloads.
ADDONS = ("prometheus", "kiali")
state = (deployments_ready("istio-system", ADDONS)
         or poll(lambda: deployments_ready("istio-system", ADDONS), timeout=150))
print("both up" if state else "still starting -- the next cell keeps waiting")

In [ ]:
state = (deployments_ready("istio-system", ADDONS)
         or poll(lambda: deployments_ready("istio-system", ADDONS), timeout=170))
assert state, (
    "Kiali or its Prometheus never became Ready -- `kubectl get pods -n istio-system` "
    "will say why. On a tight cluster this is where memory runs out."
)
print("✅ Kiali and its Prometheus backing store are up")

In [ ]:
# Same background-process rule as notebook 07: `!... &` dies with the cell's
# subshell, so use subprocess.Popen.
import subprocess, time

kiali_pf = subprocess.Popen(
    ["kubectl", "port-forward", "svc/kiali", "-n", "istio-system", "20001:20001"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print("Kiali: http://localhost:20001")

## 8) Generate traffic and view the topology

Kiali becomes more interesting when there is real traffic to visualize. We will port-forward the API gateway locally and send several requests through it.

### 🧪 Practical Exercise
Open Kiali at `http://localhost:20001`, go to the graph view for `k8s-lab`, and look for edges between services. Can you find the path from `api-gateway` to `user-service`?

Kiali's graph is built from Prometheus data, so give it 30–60 seconds after the traffic
loop finishes and set the time range to **Last 5m**. An empty graph almost always means
"no traffic in the selected window", not "the mesh is broken".

In [ ]:
import urllib.request


def request_counts():
    """How many /users requests each user-service pod has served.

    uvicorn logs one access line per request, so counting those lines per pod is
    a dependency-free way to see where traffic actually went. (Istio's own
    `istio_requests_total` metric would work too, and is what Kiali uses.)"""
    counts = {}
    for pod in ready_pods("app=user-service") + ready_pods("app=user-service,version=v2"):
        name = pod["metadata"]["name"]
        if name in counts:
            continue
        logs = subprocess.run(
            ["kubectl", "logs", name, "-n", NS, "-c", "user-service"],
            capture_output=True, text=True).stdout
        counts[name] = (pod["metadata"]["labels"].get("version"),
                        logs.count('"GET /users HTTP/1.1"'))
    return counts


before = request_counts()

gw_pf = subprocess.Popen(
    ["kubectl", "port-forward", "svc/api-gateway", "-n", "k8s-lab", "8000:8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)

# /api/users is the gateway route that actually calls user-service. The path
# `/users/1` does not exist on api-gateway -- it would 404 at the gateway and
# never produce an api-gateway -> user-service edge in the Kiali graph.
ok = err = 0
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/users", timeout=5).read()
        ok += 1
    except Exception:
        err += 1
    time.sleep(0.1)

print(f"requests: {ok} ok, {err} failed")
assert ok > 100, f"only {ok} of 120 requests succeeded through the mesh"

after = request_counts()
delta = {name: (v[0], v[1] - before.get(name, (None, 0))[1]) for name, v in after.items()}
print("\nrequests served, per pod:")
for name, (version, n) in sorted(delta.items()):
    print(f"  {name:<40} {version}  {n}")

v1 = sum(n for version, n in delta.values() if version == "v1")
v2 = sum(n for version, n in delta.values() if version == "v2")
print(f"\nv1 got {v1}, v2 got {v2}  (VirtualService weights: 90 / 10)")

# The canary is the whole point of the section, so prove it happened. The bounds
# are deliberately wide: 10% of ~120 requests is a small sample and the split is
# probabilistic per request, not a strict round-robin.
assert v1 + v2 >= ok * 0.9, \
    f"only {v1 + v2} of {ok} requests reached a user-service pod"
assert v2 > 0, \
    "the v2 subset received NO traffic -- the 10% weight is not being applied"
assert v1 > v2 * 2, \
    f"v1 should carry the large majority at 90/10, got v1={v1} v2={v2}"
print("\n✅ traffic really was split between two versions, without touching the app")

!kubectl get pods -n k8s-lab

## 🧹 Clean Up

When you finish the lab, remove Istio so the cluster returns to a simpler state.

### 🧪 Practical Exercise
After uninstalling Istio, inspect the `k8s-lab` pods again. What do you expect to happen to the sidecars and the READY counts over time?

In [ ]:
# 1. Remove the mesh configuration we created (istioctl uninstall does not
#    delete VirtualServices, DestinationRules or PeerAuthentications).
!kubectl delete -f ./istio-traffic-policy.yaml --ignore-not-found
!kubectl delete -f ./istio-canary.yaml --ignore-not-found
# ...and by name too, in case an interrupted run left one behind without the file.
!kubectl delete deployment user-service-v2 -n k8s-lab --ignore-not-found
!kubectl delete -f ./istio-peer-auth.yaml --ignore-not-found

# 2. Stop the port-forwards.
for _n in ("kiali_pf", "gw_pf"):
    _p = globals().get(_n)
    if _p is not None:
        _p.terminate()

# 3. Remove the addons, then the control plane.
!kubectl delete -f $ISTIO_DIR/samples/addons/kiali.yaml --ignore-not-found
!kubectl delete -f $ISTIO_DIR/samples/addons/prometheus.yaml --ignore-not-found
!$ISTIOCTL uninstall --purge -y

# 4. Turn injection off. Removing the label alone changes nothing about pods that
#    already exist -- they keep their sidecars until they are replaced.
!kubectl label namespace k8s-lab istio-injection-

# 5. Put Pod Security back to `baseline`. Notebooks 9 and 10 inherit this
#    namespace, and leaving it at `privileged` would quietly disable the
#    protection notebook 06 spends a section on.
!kubectl label namespace k8s-lab pod-security.kubernetes.io/enforce=baseline --overwrite

!rm -f ./istio-peer-auth.yaml ./istio-canary.yaml ./istio-traffic-policy.yaml

In [ ]:
# 6. Replace the pods so the sidecars actually go away, and undo the version
#    label the canary section added. Same two-cell wait as the injection above:
#    six pods at maxUnavailable: 0 takes longer than one cell should block for.
!kubectl patch deployment user-service -n k8s-lab --type json -p '[{"op":"remove","path":"/spec/template/metadata/labels/version"}]' || true
!kubectl rollout restart deployment -n k8s-lab

print("\nremoving sidecars:")
finished = await_sidecars(want_injected=False, budget=150)
print("done" if finished else "still rolling -- the next cell keeps waiting")

In [ ]:
injected, total = sidecar_state()
if not (total == 6 and injected == 0):
    assert await_sidecars(want_injected=False, budget=170), \
        f"sidecars are still attached: {sidecar_state()}"

print()
# Back to 1/1 -- no sidecars.
!kubectl get pods -n k8s-lab

# The hand-off to notebooks 9 and 10, checked here rather than discovered there.
# A half-finished Istio cleanup is the nastiest thing this notebook can leave
# behind: the namespace keeps `istio-injection=enabled`, every pod created
# afterwards gets an istio-init container, and with `enforce: baseline` back in
# place every one of them is refused at admission -- in a notebook that never
# mentions Istio.
labels = kget("namespace", NS, ns=None)["metadata"]["labels"]
assert "istio-injection" not in labels, \
    f"sidecar injection is still enabled on {NS}: {labels}"
assert labels.get("pod-security.kubernetes.io/enforce") == "baseline", \
    f"Pod Security was not restored to baseline: {labels}"

for name in ("api-gateway", "user-service", "order-service"):
    running = ready_pods(f"app={name}")
    assert len(running) == 2, f"{name} has {len(running)}/2 ready pods after cleanup"
    for pod in running:
        assert "istio-proxy" not in container_names(pod), \
            f"{pod['metadata']['name']} still has a sidecar"

assert not kget("pods", "-l", "version=v2")["items"], "the v2 canary pods are still around"
print("✅ mesh removed, namespace restored to baseline, 3 deployments at 2/2 with no sidecars")

## 🎓 What You Learned

Great job. In this notebook, you learned that:

- a service mesh adds a dedicated communication layer between services
- Istio uses sidecar proxies to provide mesh features
- mTLS can encrypt and authenticate service-to-service traffic
- traffic splitting supports safer canary rollouts
- retries and timeouts can be configured in mesh policy instead of app code
- Kiali helps you visualize the service graph and traffic flow — and it reads that graph
  from Prometheus, so the Prometheus addon has to be installed for it to show anything
- Sidecar injection is a **namespace label plus a pod restart**; the label alone changes
  nothing about pods that already exist, and removing it does not remove existing sidecars
- On Kubernetes 1.29+ the proxy is injected as a **native sidecar** — an entry in
  `spec.initContainers` with `restartPolicy: Always` — so it starts before your app and
  is stopped when your app exits. Anything that looks for it in `spec.containers` will
  wrongly conclude there is no sidecar
- Istio's default `istio-init` container needs `NET_ADMIN`/`NET_RAW` to rewrite iptables,
  which a namespace enforcing Pod Security `baseline` refuses. The lab relaxes the label
  and puts it back; the production fix is the **Istio CNI plugin**, which moves that work
  to a per-node DaemonSet so workload pods need no elevated capabilities at all
- `PeerAuthentication: STRICT` rejects plaintext, which locks out any client without a
  sidecar — `PERMISSIVE` is the migration path
- A `VirtualService` subset only works if a `DestinationRule` defines that subset;
  otherwise Envoy answers 503

If you can explain why sidecars make networking features reusable across many services, you understand the core value of a service mesh.